# Phase 20A — Streamlit demo

Runs `app.py` (the Router-backed NL-to-query UI built on Mac) on a Colab GPU, where the trained checkpoints and Spider databases already live on Drive. The app itself is unchanged — this notebook just gets the same repo state a GPU session needs, then exposes the Streamlit port publicly via a `cloudflared` quick tunnel, since Colab doesn't let you hit `localhost:8501` directly.

Same checkpoint layout as `phase18_eval_ablation_res.ipynb` — if your Drive folder differs from `codegen/checkpoints/{sar,generator}_{sql,nosql}`, adjust the `DRIVE` path below.

## 0. Check GPU

In [14]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"GPU {i}: {name}  ({total_gb:.1f} GB)")
        # Mirrors the 20GiB cut in src/generator/infer.py -- keep the two in sync.
        if total_gb >= 20:
            print("  -> >=20GiB (A100/L4-class). GeneratorInfer loads full bf16, no quantization.")
        else:
            print("  -> <20GiB (T4-class). GeneratorInfer loads int8 via bitsandbytes.")
else:
    raise RuntimeError("No GPU visible -- set Runtime > Change runtime type > GPU before continuing.")

GPU 0: NVIDIA A100-SXM4-40GB  (39.5 GB)
  -> >=20GiB (A100/L4-class). GeneratorInfer loads full bf16, no quantization.


## 1. Clone repo + install dependencies

In [2]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv pymongo bitsandbytes streamlit langgraph

Cloning into 'Codegen'...
remote: Enumerating objects: 1278, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 1278 (delta 44), reused 60 (delta 30), pack-reused 1178 (from 2)
Receiving objects: 100% (1278/1278), 18.03 MiB | 18.24 MiB/s, done.
Resolving deltas: 100% (903/903), done.
/content/Codegen
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 122.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 70.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 158.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [11]:
!pip install -q bm25s PyStemmer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 752.4/752.4 kB 64.2 MB/s eta 0:00:00


## 2. Mount Drive and load checkpoints (both tracks)

Loads SAR + Generator checkpoints for **both** SQL and NoSQL from Drive. Same symlink trick as Phase 18's eval notebook, so `app.py`'s `configs/config.yaml` paths (`models/sar_sql`, `models/generator_sql`, ...) resolve without any code changes.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)

for name in ['sar_sql', 'generator_sql', 'sar_nosql', 'generator_nosql']:
    dst = f'models/{name}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/checkpoints/{name}', dst)

!ls -la models/sar_sql models/generator_sql models/sar_nosql models/generator_nosql

Mounted at /content/drive
lrwxrwxrwx 1 root root 58 Aug  2 05:10 models/generator_nosql -> /content/drive/MyDrive/codegen/checkpoints/generator_nosql
lrwxrwxrwx 1 root root 56 Aug  2 05:10 models/generator_sql -> /content/drive/MyDrive/codegen/checkpoints/generator_sql
lrwxrwxrwx 1 root root 52 Aug  2 05:10 models/sar_nosql -> /content/drive/MyDrive/codegen/checkpoints/sar_nosql
lrwxrwxrwx 1 root root 50 Aug  2 05:10 models/sar_sql -> /content/drive/MyDrive/codegen/checkpoints/sar_sql


In [4]:
# Safety net: force sar.backend to memory. ChromaDB's PersistentClient can't open
# an index over a Google Drive FUSE mount, so this avoids that failure mode entirely.
text = open('configs/config.yaml').read()
text = text.replace('backend: chroma', 'backend: memory')
open('configs/config.yaml', 'w').write(text)
!grep -A1 "^sar:" configs/config.yaml | head -3

sar:
  # "memory" → SARRetriever: re-encodes corpus at startup (~30 sec). No ChromaDB needed.


## 3. Spider SQLite databases

Needed for the SQL track to *execute* (not just generate) a query — `app.py`'s DB picker reads `Data/Spider/database/` the same way `scripts/run_baseline.py` does. Uploaded once as a zip to Drive (same artifact Phase 18 uses).

In [5]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l

166


## 4. MongoDB setup (NoSQL track only)

Skip this section if you only plan to demo the SQL track. `Data/mongodb/*.json` schema-cache files are git-tracked from an earlier run on a different machine — `convert_all()` treats their existence as "already converted" and will silently skip real data insertion into this fresh session's empty `mongod`, so they're cleared first.

In [6]:
!apt-get install -y mongodb >/dev/null 2>&1 || (curl -fsSL https://pgp.mongodb.com/server-7.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor && echo "deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list && apt-get update -qq && apt-get install -y mongodb-org)
!mkdir -p /data/db
import subprocess, time
subprocess.Popen(['mongod', '--dbpath', '/data/db', '--logpath', '/var/log/mongod.log', '--fork'])
time.sleep(3)
!tail -n 5 /var/log/mongod.log

deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
The following NEW packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
0 upgraded, 9 newly installed, 0 to remove and 166 not upgraded.
Need to get 189 MB of archives.
After this operation,

In [7]:
import shutil
shutil.rmtree('Data/mongodb', ignore_errors=True)   # force a real reconversion, see note above

from src.mongodb_converter import convert_all
convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",
)

[1/166] academic — 15 collections, 42 fields
[2/166] activity_1 — 5 collections, 22 fields
[3/166] aircraft — 5 collections, 28 fields
[4/166] allergy_1 — 3 collections, 12 fields
[5/166] apartment_rentals — 6 collections, 31 fields
[6/166] architecture — 3 collections, 17 fields
[7/166] assets_maintenance — 14 collections, 64 fields
[8/166] baseball_1 — 26 collections, 352 fields
[9/166] battle_death — 3 collections, 18 fields
[10/166] behavior_monitoring — 11 collections, 64 fields
[11/166] bike_1 — 4 collections, 46 fields
[12/166] body_builder — 2 collections, 11 fields
[13/166] book_2 — 2 collections, 9 fields
[14/166] browser_web — 3 collections, 11 fields
[15/166] candidate_poll — 2 collections, 14 fields
[16/166] car_1 — 6 collections, 23 fields
[17/166] chinook_1 — 11 collections, 64 fields
[18/166] cinema — 3 collections, 17 fields
[19/166] city_record — 4 collections, 27 fields
[20/166] climbing — 2 collections, 12 fields
[21/166] club_1 — 3 collections, 15 fields
[22/166] c

## 5. DeepSeek API key

Needed by `SchemaLinker` (API mode) every time `app.py` routes a question. Paste your real key in place of the placeholder, then re-run this cell — it will not overwrite a key that's already there. **Never commit this cell with a real key filled in.**

In [15]:
from pathlib import Path

PLACEHOLDER = 'sk-71509c0b38a2481784f5ddabee30a94b'
env = Path('.env')
existing = env.read_text() if env.exists() else ''

if 'DEEPSEEK_API_KEY=' in existing and PLACEHOLDER not in existing:
    print('.env already contains a DEEPSEEK_API_KEY -- left untouched.')
else:
    env.write_text(f'DEEPSEEK_API_KEY={PLACEHOLDER}\n')
    print('Wrote .env with a placeholder. Edit it (or this cell) with your real key, then re-run.')

Wrote .env with a placeholder. Edit it (or this cell) with your real key, then re-run.


In [2]:
!pkill -f "streamlit run app.py"
!pkill -f cloudflared


## 6. Launch Streamlit + expose it publicly

Colab has no direct access to `localhost:8501`, so a `cloudflared` quick tunnel proxies it to a public `*.trycloudflare.com` URL. Unlike `localtunnel`, it has no browser interstitial page in front of it — that interstitial is what was serving HTML in place of Streamlit's JS chunks and causing the `TypeError: Importing a module script failed` error, so this avoids that failure mode entirely.

The last cell runs in the foreground and stays busy on purpose (that's what keeps the tunnel alive) — use **Runtime > Interrupt execution** to stop the demo.

In [24]:
import subprocess, socket, time

# Kill any previous run from an earlier attempt at this cell, so we don't end up
# with two streamlit processes fighting over the port.
subprocess.run(["pkill", "-f", "streamlit run app.py"], stderr=subprocess.DEVNULL)
time.sleep(1)

log = open('/content/streamlit.log', 'w')
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.headless", "true",
     "--server.address", "127.0.0.1", "--server.port", "8501"],
    stdout=log, stderr=subprocess.STDOUT,
)

# Poll the port instead of a fixed sleep -- model/import overhead can push
# startup past a fixed few-second wait, which is what left cloudflared dialing
# a port nothing was listening on yet (connection refused).
for _ in range(60):
    try:
        with socket.create_connection(("127.0.0.1", 8501), timeout=1):
            print("Streamlit is up on 127.0.0.1:8501")
            break
    except OSError:
        time.sleep(2)
else:
    print("Streamlit did not come up after 120s -- check the log below for a crash:")

!tail -n 40 /content/streamlit.log

Streamlit is up on 127.0.0.1:8501


2026-08-02 06:07:13.526 Uvicorn server started on 127.0.0.1:8501

  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:8501



In [ ]:
# Stays running -- prints "Your quick Tunnel has been created! Visit it at ...
# https://xxxx.trycloudflare.com". Open that URL directly, no password/interstitial
# step needed. Interrupt this cell to stop.
#
# Explicitly 127.0.0.1, not localhost -- on some Colab images "localhost" resolves
# to IPv6 [::1] first, which Streamlit doesn't bind by default, so cloudflared's
# dial fails with "connection refused" even though the app is up on IPv4.
!npx --yes cloudflared tunnel --url http://127.0.0.1:8501

⠙⠹⠸⠼⠴⠦2026-08-02T06:07:17Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-02T06:07:17Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-02T06:07:24Z INF +--------------------------------------------------------------------------------------------+
2026-08-02T06:07:24Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-02T06:07:24Z INF |  https://hobby-guidance-knowledge-swim.trycloudf

In [18]:
import torch
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")


cuda: True
gpu: NVIDIA A100-SXM4-40GB


In [19]:
!nvidia-smi


Sun Aug  2 06:01:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             47W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [20]:
!grep -i "running on" /content/streamlit.log


Running on: cuda


In [21]:
!grep -iE "schemalinker|attempt|error" /content/streamlit.log | tail -20


In [ ]:
!grep -c "your_actual_key_here" .env


0


In [23]:
!grep -iE "schemalinker|running on|error" /content/streamlit.log | tail -20


Running on: cuda
